In [3]:
%pip install netcdf4

import xarray as xr
import pandas as pd

import matplotlib.pyplot as plt

import os
buoy_path2 = os.path.expanduser('~/Downloads/emodnet/6200200.nc')
model_path2 = os.path.expanduser('~/Downloads/emodnet/ww3_6200200.nc')

def _find_var(ds, candidates):
    for c in candidates:
        if c in ds.variables:
            return ds[c]
    raise KeyError(f"None of {candidates} found in dataset vars: {list(ds.variables)}")

def load_hs(path):
    ds = xr.open_dataset(path, decode_times=True)
    hs = _find_var(ds, ['hs','Hs','Hsig','hsig','significant_wave_height','VHM0','swh'])
    # pick time coordinate
    if 'time' in hs.coords:
        hs = hs.swap_dims({list(hs.dims)[0]:'time'}) if list(hs.dims)[0] != 'time' else hs
        hs = hs.sortby('time')
    else:
        # attempt auto-detect a datetime coord
        for c in hs.coords:
            if 'time' in c.lower():
                hs = hs.rename({c:'time'}).sortby('time')
                break
    return hs

def main():
    hs_buoy = load_hs(buoy_path2)
    hs_model = load_hs(model_path2)

    s_buoy = hs_buoy.to_series().dropna()
    s_model = hs_model.to_series().dropna()

    # Align on common timestamps (inner join)
    df = (pd.DataFrame({'buoy': s_buoy})
            .join(pd.DataFrame({'model': s_model}), how='inner')
            .dropna())

    print("Buoy times:")
    print("  start:", s_buoy.index.min(), "end:", s_buoy.index.max(), "n:", len(s_buoy))
    print("Model times:")
    print("  start:", s_model.index.min(), "end:", s_model.index.max(), "n:", len(s_model))
    print("Common times (aligned):", len(df), "first 5:", df.index[:5].tolist())

    if df.empty:
        print("No overlapping times to plot.")
        return

    mn = float(min(df.buoy.min(), df.model.min()))
    mx = float(max(df.buoy.max(), df.model.max()))

    plt.figure(figsize=(5,5))
    plt.scatter(df.buoy, df.model, s=12, alpha=0.6, edgecolor='none')
    plt.plot([mn,mx],[mn,mx],'k--', linewidth=1)
    plt.xlabel('Buoy Hs (m)')
    plt.ylabel('Model Hs (m)')
    plt.title('Significant Wave Height: Model vs Buoy')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('wave_height_comparison.png')

if __name__ == "__main__":
    main()

DEPRECATION: Loading egg at /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/six-1.17.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sniffio-1.3.1-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/rfc3986_validator-0.1.1-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading 

In [12]:
import numpy as np
arq = open('nrmse_sorted.txt', 'r')
lines = arq.readlines()
arq.close()
nrmse = [float(line.strip().split()[2]) for line in lines]  # Extract the third column (NRMSE values)
print(np.mean(nrmse))

24.38361497326203


In [21]:
import xarray as xr 
import matplotlib.pyplot as plt
import pandas as pd

buoy_path = '~/Downloads/emodnet.bak/6201008.nc'
model_path = '~/Downloads/emodnet.bak/ww3_6201008.nc'

# Open datasets with xarray and select variables
ds_emod = xr.open_dataset(buoy_path, decode_times=True)
ds_ww3 = xr.open_dataset(model_path, decode_times=True)

hs_buoy = ds_emod['VHM0']   # EMODNET variable
hs_model = ds_ww3['hs']     # WW3 variable

# Ensure time coordinate is named 'time' and sorted
def _ensure_time(da):
    if 'time' in da.coords:
        if list(da.dims)[0] != 'time':
            da = da.swap_dims({list(da.dims)[0]: 'time'})
        return da.sortby('time')
    for c in da.coords:
        if 'time' in c.lower():
            da = da.rename({c: 'time'})
            if list(da.dims)[0] != 'time':
                da = da.swap_dims({list(da.dims)[0]: 'time'})
            return da.sortby('time')
    return da

hs_buoy = _ensure_time(hs_buoy)
hs_model = _ensure_time(hs_model)

# To pandas Series indexed by time and drop NaNs
s_buoy = hs_buoy.to_series().dropna()
s_model = hs_model.to_series().dropna()

# Print time ranges
print("Buoy data:")
print("  Start:", s_buoy.index.min(), "End:", s_buoy.index.max(), "Number of points:", len(s_buoy), "\n")

print("Model data:")
print("  Start:", s_model.index.min(), "End:", s_model.index.max(), "Number of points:", len(s_model), "\n")

# Merge on common timestamps
df_ts = (pd.DataFrame({'hs_buoy': s_buoy})
         .join(pd.DataFrame({'hs_model': s_model}), how='inner')
         .dropna())

print("Matching timestamps:", len(df_ts), "\n")

if not df_ts.empty:
    # Statistics
    corr = df_ts['hs_buoy'].corr(df_ts['hs_model'])
    bias = (df_ts['hs_model'] - df_ts['hs_buoy']).mean()
    nbias = 100.0 * bias / df_ts['hs_buoy'].mean()
    rmse = np.sqrt(((df_ts['hs_model'] - df_ts['hs_buoy'])**2).mean())
    nrmse = 100.0 * rmse / df_ts['hs_buoy'].mean()

    # Plot time series
    y_max = float(max(df_ts['hs_buoy'].max(), df_ts['hs_model'].max()))
    x_min = df_ts.index.min()

    plt.figure(figsize=(12, 6))
    plt.plot(df_ts.index, df_ts['hs_buoy'], label='EMODNET (VHM0)', linewidth=1.2, alpha=0.9)
    plt.plot(df_ts.index, df_ts['hs_model'], label='WW3 (hs)', linewidth=1.2, alpha=0.9)
    plt.xlabel('Time')
    plt.ylabel('Significant Wave Height (m)')
    plt.title('Wave Height Time Series - Buoy 6201008')
    plt.legend(loc='upper right')
    plt.grid(alpha=0.3)

    txt = f"Correlation: {corr:.3f}\nNBias: {nbias:.2f}%\nNRMSE: {nrmse:.2f}%"
    plt.text(x_min, y_max * 0.95, txt, ha='left', va='top',
             bbox=dict(facecolor='wheat', alpha=0.8, edgecolor='none'))

    plt.tight_layout()
    plt.savefig('wave_height_timeseries_6201008.png', dpi=150)
    plt.show()

    print("Statistics:")
    print(f"  Correlation: {corr:.3f}")
    print(f"  NBias: {nbias:.2f}%")
    print(f"  NRMSE: {nrmse:.2f}%")
else:
    print("No overlapping times to plot.")

Buoy data:
  Start: 0.0 End: 8783.0 Number of points: 8784 

Model data:
  Start: 2020-01-01 00:00:00 End: 2020-12-31 23:00:00 Number of points: 8784 

Matching timestamps: 0 

No overlapping times to plot.
